In [1]:
import os
import pandas as pd
import numpy as np

# ตั้งค่าการแสดงผลของ Pandas ให้เห็นคอลัมน์ครบถ้วน
pd.set_option('display.max_columns', None)

# กำหนด Path สำหรับโฟลเดอร์ raw และ processed
RAW_DIR = "../datasets/raw/"
PROCESSED_DIR = "../datasets/processed/"

# สร้างโฟลเดอร์ processed อัตโนมัติถ้ายังไม่มี
os.makedirs(PROCESSED_DIR, exist_ok=True)

print("Loading raw datasets...")

# โหลด Datasets ทั้งหมด
customers = pd.read_csv(os.path.join(RAW_DIR, "olist_customers_dataset.csv"))
geolocation = pd.read_csv(os.path.join(RAW_DIR, "olist_geolocation_dataset.csv"))
order_items = pd.read_csv(os.path.join(RAW_DIR, "olist_order_items_dataset.csv"))
order_payments = pd.read_csv(os.path.join(RAW_DIR, "olist_order_payments_dataset.csv"))
order_reviews = pd.read_csv(os.path.join(RAW_DIR, "olist_order_reviews_dataset.csv"))
orders = pd.read_csv(os.path.join(RAW_DIR, "olist_orders_dataset.csv"))
products = pd.read_csv(os.path.join(RAW_DIR, "olist_products_dataset.csv"))
sellers = pd.read_csv(os.path.join(RAW_DIR, "olist_sellers_dataset.csv"))
category_translation = pd.read_csv(os.path.join(RAW_DIR, "product_category_name_translation.csv"))

print("✅ Load datasets เรียบร้อยแล้ว!")

Loading raw datasets...
✅ Load datasets เรียบร้อยแล้ว!


In [2]:
datasets = {
    "customers": customers,
    "geolocation": geolocation,
    "order_items": order_items,
    "order_payments": order_payments,
    "order_reviews": order_reviews,
    "orders": orders,
    "products": products,
    "sellers": sellers,
    "category_translation": category_translation
}

# แสดงจำนวน Row, Column และ Missing Values ของแต่ละตาราง
print(f"{'Dataset':<22} | {'Rows':<8} | {'Columns':<8} | {'Null Count'}")
print("-" * 55)
for name, df in datasets.items():
    null_sum = df.isnull().sum().sum()
    print(f"{name:<22} | {df.shape[0]:<8} | {df.shape[1]:<8} | {null_sum}")

Dataset                | Rows     | Columns  | Null Count
-------------------------------------------------------
customers              | 99441    | 5        | 0
geolocation            | 1000163  | 5        | 0
order_items            | 112650   | 7        | 0
order_payments         | 103886   | 5        | 0
order_reviews          | 99224    | 7        | 145903
orders                 | 99441    | 8        | 4908
products               | 32951    | 9        | 2448
sellers                | 3095     | 4        | 0
category_translation   | 71       | 2        | 0


In [3]:
# 1. Customers & Sellers: Convert ZIP code prefix to String (Pad zero ตัวหน้าให้ครบ 5 หลัก)
customers['customer_zip_code_prefix'] = customers['customer_zip_code_prefix'].astype(str).str.zfill(5)
sellers['seller_zip_code_prefix'] = sellers['seller_zip_code_prefix'].astype(str).str.zfill(5)
geolocation['geolocation_zip_code_prefix'] = geolocation['geolocation_zip_code_prefix'].astype(str).str.zfill(5)

# 2. Orders: Cast timestamp columns to datetime
datetime_cols = [
    'order_purchase_timestamp', 
    'order_approved_at',
    'order_delivered_carrier_date', 
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]
for col in datetime_cols:
    orders[col] = pd.to_datetime(orders[col])

# Reviews: Cast creation & answer timestamp to datetime
order_reviews['review_creation_date'] = pd.to_datetime(order_reviews['review_creation_date'])
order_reviews['review_answer_timestamp'] = pd.to_datetime(order_reviews['review_answer_timestamp'])

# 3. Products: Translate categories and handle missing values
# - Map ภาษาอังกฤษจาก category_translation
products = products.merge(category_translation, on='product_category_name', how='left')

# - เติมค่าว่างใน category ด้วย 'unknown'
products['product_category_name'] = products['product_category_name'].fillna('unknown')
products['product_category_name_english'] = products['product_category_name_english'].fillna('unknown')

# - เติมค่าว่างของขนาด/น้ำหนักสินค้า ด้วย Median (ตาม Business Rule ในแผน)
dimension_cols = ['product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']
for col in dimension_cols:
    products[col] = products[col].fillna(products[col].median())

# 4. Geolocation: Deduplicate ZIP codes โดยการหาค่าเฉลี่ย Lat/Lng เพื่อเตรียมทำ Star Schema
geolocation_clean = geolocation.groupby('geolocation_zip_code_prefix').agg({
    'geolocation_lat': 'mean',
    'geolocation_lng': 'mean',
    'geolocation_city': 'first',
    'geolocation_state': 'first'
}).reset_index()

print("✅ Data cleaning & transformation สำเร็จเรียบร้อย!")

✅ Data cleaning & transformation สำเร็จเรียบร้อย!


In [4]:
# ตรวจสอบ Missing Values อีกครั้งหลัง Cleaning
print("--- Check Null Values After Cleaning ---")
print(f"Products missing category: {products['product_category_name_english'].isnull().sum()}")
print(f"Products missing weight: {products['product_weight_g'].isnull().sum()}")
print(f"Orders schema info (order_purchase_timestamp type): {orders['order_purchase_timestamp'].dtype}")
print(f"Customer ZIP sample: {customers['customer_zip_code_prefix'].iloc[0]} (Type: {type(customers['customer_zip_code_prefix'].iloc[0])})")
print(f"Geolocation deduplicated rows: {geolocation_clean.shape[0]:,} rows")

--- Check Null Values After Cleaning ---
Products missing category: 0
Products missing weight: 0
Orders schema info (order_purchase_timestamp type): datetime64[ns]
Customer ZIP sample: 14409 (Type: <class 'str'>)
Geolocation deduplicated rows: 19,015 rows


In [5]:
# ---------------------------------------------------------
# 1. Exploratory Statistical Analysis (EDA)
# ---------------------------------------------------------
print("=== STATISTICAL OVERVIEW ===")

# คำนวณยอดขายรวม และค่าจัดส่งรวม
total_item_revenue = order_items['price'].sum()
total_freight = order_items['freight_value'].sum()
total_gmv = total_item_revenue + total_freight

print(f"Total GMV (Sales + Freight): ${total_gmv:,.2f}")
print(f"  - Total Product Sales:     ${total_item_revenue:,.2f}")
print(f"  - Total Freight Value:    ${total_freight:,.2f}")

# สถิติของราคาและค่าส่ง
print("\n--- Price & Freight Value Summary ---")
display(order_items[['price', 'freight_value']].describe().round(2))

# สถิติ Review Score
avg_review = order_reviews['review_score'].mean()
print(f"\nAverage Review Score: {avg_review:.2f} / 5.00")

# สถิติระยะเวลาจัดส่ง (วัน) สำหรับสถานะ delivered
delivered_orders = orders[orders['order_status'] == 'delivered'].copy()
delivered_orders['delivery_days'] = (delivered_orders['order_delivered_customer_date'] - delivered_orders['order_purchase_timestamp']).dt.total_seconds() / (24 * 3600)

print(f"Average Delivery Latency: {delivered_orders['delivery_days'].mean():.1f} days")
print(f"Median Delivery Latency:  {delivered_orders['delivery_days'].median():.1f} days")

=== STATISTICAL OVERVIEW ===
Total GMV (Sales + Freight): $15,843,553.24
  - Total Product Sales:     $13,591,643.70
  - Total Freight Value:    $2,251,909.54

--- Price & Freight Value Summary ---


,price,freight_value
count,112650.00,112650.00
mean,120.65,19.99
std,183.63,15.81
min,0.85,0.00
25%,39.90,13.08
50%,74.99,16.26
75%,134.90,21.15
max,6735.00,409.68



Average Review Score: 4.09 / 5.00
Average Delivery Latency: 12.6 days
Median Delivery Latency:  10.2 days


In [6]:
# ---------------------------------------------------------
# 2. Export Cleaned Datasets to /datasets/processed
# ---------------------------------------------------------
print("Exporting cleaned files to datasets/processed/ ...")

customers.to_csv(os.path.join(PROCESSED_DIR, "dim_customers.csv"), index=False)
geolocation_clean.to_csv(os.path.join(PROCESSED_DIR, "dim_geolocation.csv"), index=False)
products.to_csv(os.path.join(PROCESSED_DIR, "dim_products.csv"), index=False)
sellers.to_csv(os.path.join(PROCESSED_DIR, "dim_sellers.csv"), index=False)
orders.to_csv(os.path.join(PROCESSED_DIR, "fact_orders.csv"), index=False)
order_items.to_csv(os.path.join(PROCESSED_DIR, "fact_order_items.csv"), index=False)
order_payments.to_csv(os.path.join(PROCESSED_DIR, "fact_order_payments.csv"), index=False)
order_reviews.to_csv(os.path.join(PROCESSED_DIR, "fact_order_reviews.csv"), index=False)

print("✅ Export Processed CSV files เรียบร้อยแล้ว!")

Exporting cleaned files to datasets/processed/ ...
✅ Export Processed CSV files เรียบร้อยแล้ว!


In [7]:
from sqlalchemy import create_engine

# ---------------------------------------------------------
# 1. Database Connection Configuration
# ---------------------------------------------------------
DB_USER = "postgres"        # Username ของ PostgreSQL
DB_PASSWORD = "your_password" # Password ของคุณ
DB_HOST = "localhost"        # Host (ปกติคือ localhost)
DB_PORT = "5432"             # Port (ปกติคือ 5432)
DB_NAME = "olist_db"         # ชื่อ Database ที่สร้างไว้ใน PostgreSQL

# Create Connection Engine
connection_string = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(connection_string)

# ---------------------------------------------------------
# 2. Define Tables Mapping
# ---------------------------------------------------------
tables_to_load = {
    "dim_customers.csv": "customers",
    "dim_geolocation.csv": "geolocation",
    "dim_products.csv": "products",
    "dim_sellers.csv": "sellers",
    "fact_orders.csv": "orders",
    "fact_order_items.csv": "order_items",
    "fact_order_payments.csv": "payments",
    "fact_order_reviews.csv": "reviews"
}

# ---------------------------------------------------------
# 3. Upload Processed Data to Database
# ---------------------------------------------------------
print("Starting database upload...")

for file_name, table_name in tables_to_load.items():
    file_path = os.path.join(PROCESSED_DIR, file_name)
    df = pd.read_csv(file_path)
    
    # Import เข้า Database (ถ้ามีตารางเดิมอยู่แล้วจะ Replace ใหม่)
    df.to_sql(name=table_name, con=engine, if_exists='replace', index=False)
    print(f"  - Uploaded {table_name:<12} ({df.shape[0]:,} rows)")

print("\n✅ Upload all processed datasets to PostgreSQL successfully!")

Starting database upload...
  - Uploaded customers    (99,441 rows)
  - Uploaded geolocation  (19,015 rows)
  - Uploaded products     (32,951 rows)
  - Uploaded sellers      (3,095 rows)
  - Uploaded orders       (99,441 rows)
  - Uploaded order_items  (112,650 rows)
  - Uploaded payments     (103,886 rows)
  - Uploaded reviews      (99,224 rows)

✅ Upload all processed datasets to PostgreSQL successfully!
